# Aula 03: Fases do Projeto de Banco de Dados e Introdução ao Modelo Entidade-Relacionamento (MER)
**Disciplina:** Banco de Dados  
**Curso:** Tecnologia em Telemática -- IFCE Campus Tauá  
**Professor:** Prof. Me. Reginaldo Pereira Fernandes  

---
## 🎯 Objetivos do Laboratório
1. Compreender o ciclo de vida do projeto de banco de dados (Requisitos $\to$ Conceitual $\to$ Lógico $\to$ Físico).
2. Implementar um metamodelo computacional do **Modelo Entidade-Relacionamento (Peter Chen, 1976)** em Python.
3. Classificar e validar todos os tipos canônicos de atributos:
   - **Identificador (Chave Primária Conceitual)**;
   - **Simples / Atômicos**;
   - **Compostos** (hierarquia de subatributos);
   - **Multivalorados** (conjuntos de valores);
   - **Armazenados vs. Derivados** (cálculo dinâmico em tempo de execução).
4. Demonstrar a transição pré-relacional em SQLite respeitando a **Primeira Forma Normal (1FN)**.

### 1. Metamodelo Conceitual do MER em Python
Definimos as classes que representam o metamodelo do MER: `TipoAtributo`, `DefinicaoAtributo` e `EsquemaEntidade`.

In [1]:
from enum import Enum
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Callable
import sqlite3

class TipoAtributo(Enum):
    IDENTIFICADOR = "Chave Identificadora (Sublinhado no DER)"
    SIMPLES = "Simples / Atômico (Elipse simples no DER)"
    COMPOSTO = "Composto (Elipse ramificada no DER)"
    MULTIVALORADO = "Multivalorado (Elipse dupla no DER)"
    DERIVADO = "Derivado (Elipse tracejada no DER)"

class DefinicaoAtributo:
    def __init__(self, nome: str, tipo: TipoAtributo, descricao: str,
                 sub_atributos: Optional[List['DefinicaoAtributo']] = None,
                 funcao_derivacao: Optional[Callable[[Dict[str, Any]], Any]] = None,
                 obrigatorio: bool = True):
        self.nome = nome
        self.tipo = tipo
        self.descricao = descricao
        self.sub_atributos = sub_atributos or []
        self.funcao_derivacao = funcao_derivacao
        self.obrigatorio = obrigatorio

    def __repr__(self) -> str:
        return f"<Atributo '{self.nome}' ({self.tipo.name})>"

class EsquemaEntidade:
    def __init__(self, nome: str, descricao: str):
        self.nome = nome
        self.descricao = descricao
        self.atributos: Dict[str, DefinicaoAtributo] = {}
        self.instancias: List[Dict[str, Any]] = []

    def adicionar_atributo(self, atributo: DefinicaoAtributo):
        self.atributos[atributo.nome] = atributo

    def validar_e_inserir(self, dados_instancia: Dict[str, Any]):
        # 1. Validar identificador unico
        id_attr = next((a for a in self.atributos.values() if a.tipo == TipoAtributo.IDENTIFICADOR), None)
        if id_attr:
            val_id = dados_instancia.get(id_attr.nome)
            if val_id is None:
                raise ValueError(f"Atributo identificador '{id_attr.nome}' nao pode ser nulo.")
            for inst in self.instancias:
                if inst.get(id_attr.nome) == val_id:
                    raise ValueError(f"Violacao de unicidade: Chave '{val_id}' ja existe.")

        # 2. Validar atributos obrigatorios
        for nome_attr, attr in self.atributos.items():
            if attr.tipo != TipoAtributo.DERIVADO and attr.obrigatorio:
                if nome_attr not in dados_instancia or dados_instancia[nome_attr] is None:
                    raise ValueError(f"Atributo obrigatorio '{nome_attr}' ausente.")

        # 3. Computar atributos derivados dinamicamente
        for nome_attr, attr in self.atributos.items():
            if attr.tipo == TipoAtributo.DERIVADO and attr.funcao_derivacao:
                dados_instancia[nome_attr] = attr.funcao_derivacao(dados_instancia)

        self.instancias.append(dados_instancia)

print("✅ Metamodelo do MER carregado com sucesso!")

### 2. Especificação do Minimundo: Entidade `DISPOSITIVO_REDE`
Configuramos a entidade com todos os tipos canônicos de atributos.

In [2]:
esquema = EsquemaEntidade(
    nome="DISPOSITIVO_REDE",
    descricao="Equipamentos ativos da infraestrutura de telecomunicações do campus."
)

# Identificador
esquema.adicionar_atributo(DefinicaoAtributo("patrimonio_id", TipoAtributo.IDENTIFICADOR, "Chave primaria conceitual"))

# Simples / Atomicos
esquema.adicionar_atributo(DefinicaoAtributo("hostname", TipoAtributo.SIMPLES, "Nome na topologia"))
esquema.adicionar_atributo(DefinicaoAtributo("ip_loopback", TipoAtributo.SIMPLES, "IPv4 de gerencia loopback0"))
esquema.adicionar_atributo(DefinicaoAtributo("fabricante", TipoAtributo.SIMPLES, "Fabricante do hardware"))
esquema.adicionar_atributo(DefinicaoAtributo("data_instalacao", TipoAtributo.SIMPLES, "Data de ativacao fisica"))

# Composto
sub_rack = [
    DefinicaoAtributo("bloco", TipoAtributo.SIMPLES, "Bloco predial"),
    DefinicaoAtributo("sala_noc", TipoAtributo.SIMPLES, "Sala do NOC"),
    DefinicaoAtributo("rack_id", TipoAtributo.SIMPLES, "ID do Rack"),
    DefinicaoAtributo("posicao_u", TipoAtributo.SIMPLES, "Unidade U")
]
esquema.adicionar_atributo(DefinicaoAtributo("localizacao_rack", TipoAtributo.COMPOSTO, "Localizacao física", sub_atributos=sub_rack))

# Multivalorado
esquema.adicionar_atributo(DefinicaoAtributo("vlan_tags", TipoAtributo.MULTIVALORADO, "Tags IEEE 802.1Q"))

# Derivado
def calcular_dias_operacao(dados: Dict[str, Any]) -> int:
    dt_inst = datetime.strptime(dados["data_instalacao"], "%Y-%m-%d").date()
    return (date(2026, 10, 6) - dt_inst).days

esquema.adicionar_atributo(DefinicaoAtributo(
    nome="dias_operacao",
    tipo=TipoAtributo.DERIVADO,
    descricao="Dias de operacao ininterrupta",
    funcao_derivacao=calcular_dias_operacao,
    obrigatorio=False
))

print(f"✅ Esquema concebido: {len(esquema.atributos)} atributos registrados no conjunto {esquema.nome}.")

### 3. Validação Semântica, Instanciação e Cálculo Dinâmico
Inserimos entidades reais do campus com verificação de integridade e cálculo do atributo derivado.

In [3]:
roteador_core = {
    "patrimonio_id": "IFCE-TAU-00101",
    "hostname": "TAU-RTR-CORE-01",
    "ip_loopback": "10.200.0.1",
    "fabricante": "Cisco Systems",
    "data_instalacao": "2024-02-15",
    "localizacao_rack": {
        "bloco": "Bloco Principal",
        "sala_noc": "NOC Sala 102",
        "rack_id": "RACK-01",
        "posicao_u": "U40-U42"
    },
    "vlan_tags": [10, 20, 30, 99, 100, 200]
}

switch_dist = {
    "patrimonio_id": "IFCE-TAU-00102",
    "hostname": "TAU-SW-DIST-01",
    "ip_loopback": "10.200.0.2",
    "fabricante": "Huawei",
    "data_instalacao": "2025-05-10",
    "localizacao_rack": {
        "bloco": "Bloco Didático",
        "sala_noc": "Armário Telecom 02",
        "rack_id": "RACK-03",
        "posicao_u": "U22-U23"
    },
    "vlan_tags": [10, 20, 100]
}

esquema.validar_e_inserir(roteador_core)
esquema.validar_e_inserir(switch_dist)

for inst in esquema.instancias:
    print(f"-> Host: {inst['hostname']:<16} | Chave: {inst['patrimonio_id']:<15} | Uptime: {inst['dias_operacao']} dias | VLANs: {inst['vlan_tags']}")

### 4. Transição Metodológica: Mapeamento Conceitual $\to$ Relacional no SQLite
Demonstração de como atributos compostos são planificados e atributos multivalorados geram tabelas associativas para atender à Primeira Forma Normal (1FN).

In [4]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 1. Tabela Principal (Composto achatado em colunas atômicas)
cursor.execute("""
CREATE TABLE dispositivos_rede (
    patrimonio_id TEXT PRIMARY KEY,
    hostname TEXT NOT NULL UNIQUE,
    ip_loopback TEXT NOT NULL UNIQUE,
    fabricante TEXT NOT NULL,
    data_instalacao DATE NOT NULL,
    rack_bloco TEXT NOT NULL,
    rack_sala TEXT NOT NULL,
    rack_id TEXT NOT NULL,
    rack_posicao_u TEXT NOT NULL
);
""")

# 2. Tabela Auxiliar para o Multivalorado (1FN)
cursor.execute("""
CREATE TABLE dispositivo_vlans (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    dispositivo_patrimonio TEXT NOT NULL,
    vlan_id INTEGER NOT NULL CHECK (vlan_id BETWEEN 1 AND 4094),
    FOREIGN KEY (dispositivo_patrimonio) REFERENCES dispositivos_rede(patrimonio_id) ON DELETE CASCADE
);
""")

# 3. VIEW com o Atributo Derivado
cursor.execute("""
CREATE VIEW vw_dispositivos_com_uptime AS
SELECT 
    patrimonio_id, hostname, ip_loopback, rack_id, data_instalacao,
    CAST((julianday('2026-10-06') - julianday(data_instalacao)) AS INTEGER) AS dias_operacao_derivado
FROM dispositivos_rede;
""")

# Insercao de dados
cursor.execute("""
INSERT INTO dispositivos_rede VALUES 
('IFCE-TAU-00101', 'TAU-RTR-CORE-01', '10.200.0.1', 'Cisco Systems', '2024-02-15', 'Bloco Principal', 'NOC 102', 'RACK-01', 'U40-U42'),
('IFCE-TAU-00102', 'TAU-SW-DIST-01', '10.200.0.2', 'Huawei', '2025-05-10', 'Bloco Didático', 'Telecom 02', 'RACK-03', 'U22-U23');
""")

vlans = [('IFCE-TAU-00101', 10), ('IFCE-TAU-00101', 20), ('IFCE-TAU-00101', 100), ('IFCE-TAU-00102', 10), ('IFCE-TAU-00102', 20)]
cursor.executemany("INSERT INTO dispositivo_vlans (dispositivo_patrimonio, vlan_id) VALUES (?, ?);", vlans)
conn.commit()

# Consulta DQL
cursor.execute("""
SELECT d.hostname, d.dias_operacao_derivado, GROUP_CONCAT(v.vlan_id, ', ') AS vlans
FROM vw_dispositivos_com_uptime d
JOIN dispositivo_vlans v ON d.patrimonio_id = v.dispositivo_patrimonio
GROUP BY d.patrimonio_id;
""")

for host, uptime, vlan_list in cursor.fetchall():
    print(f"-> Dispositivo: {host:<16} | Uptime: {uptime} dias | VLANs Registradas: [{vlan_list}]")

print("\n✅ Laboratório concluído com sucesso!")